***OUR ANN PROJECT ***

In [ ]:
import kagglehub
path = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces")

Using Colab cache for faster access to the '140k-real-and-fake-faces' dataset.


In [ ]:
import os, glob, shutil, subprocess, numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score

keras.utils.set_random_seed(42)
print("GPUs:", tf.config.list_physical_devices("GPU"))   # must NOT be empty

GPUs: []


In [ ]:
import os, glob
BASE = glob.glob(f"{path}/**/real-vs-fake", recursive=True)[0]
print("BASE =", BASE, os.listdir(BASE))   # should show train, valid, test

BASE = /kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake ['valid', 'test', 'train']


In [ ]:
import glob
print(path)
BASE = glob.glob(f"{path}/**/real-vs-fake", recursive=True)[0]
print("BASE =", BASE, os.listdir(BASE))   # should show train, valid, test

/kaggle/input/140k-real-and-fake-faces
BASE = /kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake ['valid', 'test', 'train']


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/deepfake"
os.makedirs(SAVE_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
IMG = (128, 128)      # use (160,160) if stuck below 95%
BS = 128
AUTOTUNE = tf.data.AUTOTUNE

def load(split, shuffle):
    ds = keras.utils.image_dataset_from_directory(
        f"{BASE}/{split}", image_size=IMG, batch_size=BS,
        label_mode="binary", shuffle=shuffle, seed=42)
    print(split, ds.class_names)   # fake=0, real=1
    return ds.prefetch(AUTOTUNE)

train_ds = load("train", True)
val_ds   = load("valid", False)
test_ds  = load("test", False)

Found 100000 files belonging to 2 classes.
train ['fake', 'real']
Found 20000 files belonging to 2 classes.
valid ['fake', 'real']
Found 20000 files belonging to 2 classes.
test ['fake', 'real']


In [ ]:
mixed_precision.set_global_policy("mixed_float16")

def res_block(x, filters, stride=1):
    shortcut = x
    x = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    return layers.ReLU()(layers.Add()([x, shortcut]))

inputs = keras.Input(IMG + (3,))
x = layers.Rescaling(1./255)(inputs)
x = layers.RandomFlip("horizontal")(x)

x = layers.Conv2D(32, 3, strides=2, padding="same", use_bias=False)(x)
x = layers.BatchNormalization()(x)
x = layers.ReLU()(x)

x = res_block(x, 32)
x = res_block(x, 64, stride=2)
x = res_block(x, 64)
x = res_block(x, 128, stride=2)
x = res_block(x, 128)
x = res_block(x, 256, stride=2)
x = res_block(x, 256)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(1, activation="sigmoid", dtype="float32")(x)
model = keras.Model(inputs, outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 128, 128,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_flip         │ (None, 128, 128,  │          0 │ rescaling[0][0]   │
│ (RandomFlip)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 64, 64,    │        864 │ random_flip[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64, 64,    │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │      9,216 │ re_lu[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        128 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 64, 64,    │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │      9,216 │ re_lu_1[0][0]     │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        128 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 64, 64,    │          0 │ batch_normalizat… │
│                     │ 32)               │            │ re_lu[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 64, 64,    │          0 │ add[0][0]         │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 32, 32,    │     18,432 │ re_lu_2[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │     36,864 │ re_lu_3[0][0]   

 Total params: 2,781,409 (10.61 MB)

 Trainable params: 2,776,737 (10.59 MB)

 Non-trainable params: 4,672 (18.25 KB)

In [ ]:
EPOCHS = 30
steps = len(train_ds) * EPOCHS
lr = keras.optimizers.schedules.CosineDecay(2e-3, decay_steps=steps)

model.compile(optimizer=keras.optimizers.AdamW(lr, weight_decay=1e-4),
              loss="binary_crossentropy", metrics=["accuracy"])

cb = [keras.callbacks.ModelCheckpoint(f"{SAVE_DIR}/best.keras",
                                      monitor="val_accuracy", save_best_only=True),
      keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6,
                                    restore_best_weights=True)]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                    callbacks=cb, verbose=2)

Epoch 1/30


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history.history["accuracy"], label="train")
ax[0].plot(history.history["val_accuracy"], label="val")
ax[0].set_title("Accuracy"); ax[0].legend()
ax[1].plot(history.history["loss"], label="train")
ax[1].plot(history.history["val_loss"], label="val")
ax[1].set_title("Loss"); ax[1].legend()
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc*100:.2f}%")

y_true = np.concatenate([y.numpy().ravel() for _, y in test_ds]).astype(int)
y_prob = model.predict(test_ds).ravel()
y_pred = (y_prob > 0.5).astype(int)
print(classification_report(y_true, y_pred, target_names=["fake", "real"], digits=4))

cm = confusion_matrix(y_true, y_pred)
fpr, tpr, _ = roc_curve(y_true, y_prob)
auc = roc_auc_score(y_true, y_prob)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(cm, cmap="Blues")
ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(["fake", "real"])
ax[0].set_yticks([0, 1]); ax[0].set_yticklabels(["fake", "real"])
ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("Actual"); ax[0].set_title("Confusion matrix")
for i in range(2):
    for j in range(2):
        ax[0].text(j, i, cm[i, j], ha="center", va="center",
                   color="white" if cm[i, j] > cm.max()/2 else "black")
ax[1].plot(fpr, tpr, label=f"AUC = {auc:.4f}")
ax[1].plot([0, 1], [0, 1], "--", color="gray")
ax[1].set_xlabel("False positive rate"); ax[1].set_ylabel("True positive rate")
ax[1].set_title("ROC curve"); ax[1].legend()
plt.show()

In [ ]:
model.save(f"{SAVE_DIR}/deepfake_detector.keras")

In [ ]:
import kagglehub
p = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
ROOT = os.path.dirname(glob.glob(f"{p}/**/train", recursive=True)[0])
print(ROOT, os.listdir(ROOT))   # should show train, test

In [ ]:
IMG2, BS2 = (32, 32), 128

def load2(split, **kw):
    return keras.utils.image_dataset_from_directory(
        f"{ROOT}/{split}", image_size=IMG2, batch_size=BS2, label_mode="binary", **kw)

train2 = load2("train", validation_split=0.1, subset="training", seed=42).prefetch(AUTOTUNE)
val2   = load2("train", validation_split=0.1, subset="validation", seed=42).prefetch(AUTOTUNE)
test2  = load2("test", shuffle=False).prefetch(AUTOTUNE)

In [ ]:
def build(img, first_stride):
    inputs = keras.Input(img + (3,))
    x = layers.Rescaling(1./255)(inputs)
    x = layers.RandomFlip("horizontal")(x)
    x = layers.Conv2D(32, 3, strides=first_stride, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    for f, s in [(32,1), (64,2), (64,1), (128,2), (128,1), (256,2), (256,1)]:
        x = res_block(x, f, s)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation="sigmoid", dtype="float32")(x)
    return keras.Model(inputs, out)

model2 = build(IMG2, 1)

In [ ]:
EPOCHS = 30
lr = keras.optimizers.schedules.CosineDecay(2e-3, decay_steps=len(train2) * EPOCHS)
model2.compile(optimizer=keras.optimizers.AdamW(lr, weight_decay=1e-4),
               loss="binary_crossentropy", metrics=["accuracy"])

cb2 = [keras.callbacks.ModelCheckpoint(f"{SAVE_DIR}/cifake_best.keras",
                                       monitor="val_accuracy", save_best_only=True),
       keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6,
                                     restore_best_weights=True)]

history2 = model2.fit(train2, validation_data=val2, epochs=EPOCHS, callbacks=cb2, verbose=2)

In [ ]:
print(model2.evaluate(test2))
y_true = np.concatenate([y.numpy().ravel() for _, y in test2]).astype(int)
y_pred = (model2.predict(test2).ravel() > 0.5).astype(int)
print(classification_report(y_true, y_pred, target_names=["fake", "real"], digits=4))
print(confusion_matrix(y_true, y_pred))
model2.save(f"{SAVE_DIR}/ai_image_detector.keras")

In [ ]:
h = history.history
acc_gap = np.array(h["accuracy"]) - np.array(h["val_accuracy"])
best = int(np.argmax(h["val_accuracy"]))
print(f"Best val acc: {h['val_accuracy'][best]*100:.2f}% at epoch {best+1}")
print(f"Final acc gap (train - val): {acc_gap[-1]*100:.2f} points")
print(f"Lowest val loss: {min(h['val_loss']):.4f} at epoch {int(np.argmin(h['val_loss']))+1}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(acc_gap * 100); ax[0].set_title("Accuracy gap (train - val), points")
ax[0].set_xlabel("epoch")
ax[1].plot(h["val_loss"]); ax[1].set_title("Validation loss (rising = overfitting)")
ax[1].set_xlabel("epoch")
plt.show()

In [ ]:
tr_loss, tr_acc = model.evaluate(train_ds.take(200), verbose=0)   # ~25k train images
va_loss, va_acc = model.evaluate(val_ds, verbose=0)
te_loss, te_acc = model.evaluate(test_ds, verbose=0)
print(f"Train: acc {tr_acc*100:.2f}%  loss {tr_loss:.4f}")
print(f"Val  : acc {va_acc*100:.2f}%  loss {va_loss:.4f}")
print(f"Test : acc {te_acc*100:.2f}%  loss {te_loss:.4f}")

In [ ]:

def jpeg_ds(ds, quality):
    def f(x, y):
        x = tf.map_fn(lambda im: tf.image.adjust_jpeg_quality(tf.cast(im, tf.uint8), quality),
                      x, fn_output_signature=tf.uint8)
        return tf.cast(x, tf.float32), y
    return ds.map(f)

for q in [95, 75, 50]:
    _, acc = model.evaluate(jpeg_ds(test_ds.take(40), q), verbose=0)   # ~5k images
    print(f"JPEG quality {q}: {acc*100:.2f}%")

In [ ]:
test_mix = load("test", True).take(40).cache()   # shuffled, so real + fake are mixed
for _ in test_mix: pass                           # fill the cache so every test uses the same images

def per_class(ds):
    y = np.concatenate([b[1].numpy().ravel() for b in ds]).astype(int)
    p = (model.predict(ds, verbose=0).ravel() > 0.5).astype(int)
    return (p[y == 0] == 0).mean() * 100, (p[y == 1] == 1).mean() * 100

fa, ra = per_class(test_mix)
print(f"No JPEG       : fake caught {fa:.1f}% | real kept real {ra:.1f}%")
for q in [95, 75, 50]:
    fa, ra = per_class(jpeg_ds(test_mix, q))
    print(f"JPEG quality {q}: fake caught {fa:.1f}% | real kept real {ra:.1f}%")

In [ ]:
def jpeg_aug(x, y):
    def one(im):
        im8 = tf.cast(im, tf.uint8)
        return tf.cond(tf.random.uniform([]) < 0.5,
                       lambda: tf.image.random_jpeg_quality(im8, 40, 95),
                       lambda: im8)
    x = tf.map_fn(one, x, fn_output_signature=tf.uint8)
    return tf.cast(x, tf.float32), y

train_aug = train_ds.map(jpeg_aug, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_aug   = val_ds.map(jpeg_aug, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

EPOCHS_FT = 8
lr = keras.optimizers.schedules.CosineDecay(5e-4, decay_steps=len(train_ds) * EPOCHS_FT)
model.compile(optimizer=keras.optimizers.AdamW(lr, weight_decay=1e-4),
              loss="binary_crossentropy", metrics=["accuracy"])

cb_ft = [keras.callbacks.ModelCheckpoint(f"{SAVE_DIR}/jpeg_robust.keras",
                                         monitor="val_accuracy", save_best_only=True),
         keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4,
                                       restore_best_weights=True)]

history_ft = model.fit(train_aug, validation_data=val_aug, epochs=EPOCHS_FT,
                       callbacks=cb_ft, verbose=2)

In [ ]:
import os, glob, subprocess, numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import kagglehub
from google.colab import drive

print("GPUs:", tf.config.list_physical_devices("GPU"))   # should NOT be empty
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/deepfake"
print("Drive files:", os.listdir(SAVE_DIR))

AUTOTUNE = tf.data.AUTOTUNE
IMG, BS = (128, 128), 128

p = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces")
BASE = glob.glob(f"{p}/**/real-vs-fake", recursive=True)[0]

def load(split, shuffle):
    ds = keras.utils.image_dataset_from_directory(
        f"{BASE}/{split}", image_size=IMG, batch_size=BS,
        label_mode="binary", shuffle=shuffle, seed=42)
    return ds.prefetch(AUTOTUNE)

test_ds = load("test", False)

# the JPEG-robust model saved by the fine-tuning run
model = keras.models.load_model(f"{SAVE_DIR}/jpeg_robust.keras", compile=False)
model.compile(loss="binary_crossentropy", metrics=["accuracy"])

def jpeg_ds(ds, quality):
    def f(x, y):
        x = tf.map_fn(lambda im: tf.image.adjust_jpeg_quality(tf.cast(im, tf.uint8), quality),
                      x, fn_output_signature=tf.uint8)
        return tf.cast(x, tf.float32), y
    return ds.map(f)

def per_class(ds):
    y = np.concatenate([b[1].numpy().ravel() for b in ds]).astype(int)
    pr = (model.predict(ds, verbose=0).ravel() > 0.5).astype(int)
    return (pr[y == 0] == 0).mean() * 100, (pr[y == 1] == 1).mean() * 100

test_mix = load("test", True).take(40).cache()
for _ in test_mix: pass

In [ ]:
subprocess.run(["pip", "install", "-q", "gradio"], check=True)
import gradio as gr
from PIL import Image

face_model = model
ai_model = keras.models.load_model(f"{SAVE_DIR}/ai_image_detector.keras", compile=False)

In [ ]:
p2 = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
ROOT = os.path.dirname(glob.glob(f"{p2}/**/train", recursive=True)[0])
test2 = keras.utils.image_dataset_from_directory(
    f"{ROOT}/test", image_size=(32, 32), batch_size=128,
    label_mode="binary", shuffle=False).prefetch(AUTOTUNE)

In [ ]:

def make_grad_model(m):
    last_relu = [l for l in m.layers if isinstance(l, layers.ReLU)][-1]
    return keras.Model(m.inputs, [last_relu.output, m.output])

MODELS = {
    "Deepfake face detector": (face_model, make_grad_model(face_model), (128, 128)),
    "AI-generated image detector": (ai_model, make_grad_model(ai_model), (32, 32)),
}

def gradcam_for(gm, arr, size):
    x = tf.expand_dims(tf.cast(arr, tf.float32), 0)
    with tf.GradientTape() as tape:
        conv, pred = gm(x, training=False)
        p = pred[:, 0]
        score = p if p[0] > 0.5 else 1.0 - p
    grads = tf.cast(tape.gradient(score, conv), tf.float32)
    conv = tf.cast(conv, tf.float32)
    w = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(w * conv, axis=-1))[0]
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return tf.image.resize(cam[..., None], size).numpy()[..., 0], float(p[0])

In [ ]:
os.makedirs("/content/demo_samples", exist_ok=True)
examples = []

# faces: 3 fake + 3 real
imgs, labels = next(iter(test_mix))
cnt = {0: 0, 1: 0}
for im, lb in zip(imgs.numpy(), labels.numpy().ravel().astype(int)):
    if cnt[lb] < 3:
        name = ["fake", "real"][lb]
        path = f"/content/demo_samples/face_{name}_{cnt[lb]}.png"
        Image.fromarray(im.astype("uint8")).save(path)
        examples.append([path, "Deepfake face detector"])
        cnt[lb] += 1

# AI-generated images (CIFAKE): 3 fake + 3 real
cnt = {0: 0, 1: 0}
for im, lb in test2.unbatch().shuffle(2000, seed=3).take(60):
    lb = int(lb.numpy()[0])
    if cnt[lb] < 3:
        name = ["fake", "real"][lb]
        path = f"/content/demo_samples/ai_{name}_{cnt[lb]}.png"
        Image.fromarray(im.numpy().astype("uint8")).save(path)
        examples.append([path, "AI-generated image detector"])
        cnt[lb] += 1
print(len(examples), "examples ready")   # should print 12

In [ ]:
imgs, labels = next(iter(test2.skip(100)))   # second half of the test set = real images
print(labels[:3].numpy().ravel())            # should print [1. 1. 1.]
for i in range(3):
    path = f"/content/demo_samples/ai_real_{i}.png"
    Image.fromarray(imgs[i].numpy().astype("uint8")).save(path)
    examples.append([path, "AI-generated image detector"])
print(len(examples), "examples ready")       # should now print 12

In [ ]:
def detect(image, which):
    if image is None:
        return None, None
    m, gm, size = MODELS[which]
    pil = Image.fromarray(image).convert("RGB")
    arr = np.array(pil.resize(size, Image.BILINEAR)).astype("float32")
    cam, p_real = gradcam_for(gm, arr, size)

    base = np.array(pil.resize((384, 384))) / 255.0
    cam_big = np.array(Image.fromarray((cam * 255).astype("uint8")).resize((384, 384))) / 255.0
    heat = plt.cm.jet(cam_big)[..., :3]
    overlay = (0.55 * base + 0.45 * heat).clip(0, 1)
    verdict = {"FAKE (AI-generated)": 1 - p_real, "REAL": p_real}
    return verdict, (overlay * 255).astype("uint8")

demo = gr.Interface(
    fn=detect,
    inputs=[gr.Image(type="numpy", label="Upload an image"),
            gr.Dropdown(list(MODELS), value="Deepfake face detector", label="Model")],
    outputs=[gr.Label(label="Verdict"),
             gr.Image(label="Where the model looked (Grad-CAM)")],
    examples=examples,
    title="Deepfake & AI-Generated Image Detector",
    description="Custom ResNet-style CNNs trained from scratch. Faces: 140k Real and Fake Faces. "
                "AI images: CIFAKE. Red areas = regions that influenced the decision.",
)
demo.launch(share=True)

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

def build_resnet(img_size, resize_to=None, name="resnet50"):
    inputs = keras.Input(img_size + (3,))
    x = inputs
    in_size = img_size
    if resize_to is not None:
        x = layers.Resizing(*resize_to)(x)
        in_size = resize_to
    x = layers.RandomFlip("horizontal")(x)
    x = preprocess_input(x)

    base = ResNet50(include_top=False, weights="imagenet",
                    input_shape=in_size + (3,), name=name)
    base.trainable = False
    x = base(x, training=False)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation="sigmoid", dtype="float32")(x)
    return keras.Model(inputs, out), base

face_resnet, face_base = build_resnet(IMG)

In [ ]:
face_resnet.compile(optimizer=keras.optimizers.Adam(1e-3),
                    loss="binary_crossentropy", metrics=["accuracy"])
h1 = face_resnet.fit(train_ds, validation_data=val_ds, epochs=3, verbose=2)

In [ ]:
EPOCHS_R = 8
face_base.trainable = True
lr = keras.optimizers.schedules.CosineDecay(1e-4, decay_steps=len(train_ds) * EPOCHS_R)
face_resnet.compile(optimizer=keras.optimizers.AdamW(lr, weight_decay=1e-4),
                    loss="binary_crossentropy", metrics=["accuracy"])

cb_r = [keras.callbacks.ModelCheckpoint(f"{SAVE_DIR}/resnet50_face_best.keras",
                                        monitor="val_accuracy", save_best_only=True),
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3,
                                      restore_best_weights=True)]

h2 = face_resnet.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_R,
                     callbacks=cb_r, verbose=2)
face_resnet.save(f"{SAVE_DIR}/resnet50_face_detector.keras")

In [ ]:
def evaluate_binary(m, ds):
    y_true = np.concatenate([y.numpy().ravel() for _, y in ds]).astype(int)
    y_prob = m.predict(ds, verbose=0).ravel()
    y_pred = (y_prob > 0.5).astype(int)
    print(classification_report(y_true, y_pred, target_names=["fake", "real"], digits=4))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
    print(f"AUC: {roc_auc_score(y_true, y_prob):.4f}")

print("ResNet50 faces test acc: {:.2f}%".format(face_resnet.evaluate(test_ds, verbose=0)[1] * 100))
evaluate_binary(face_resnet, test_ds)

In [ ]:
ai_resnet, ai_base = build_resnet(IMG2, resize_to=(128, 128))

# phase 1
ai_resnet.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy", metrics=["accuracy"])
a1 = ai_resnet.fit(train2, validation_data=val2, epochs=3, verbose=2)

# phase 2
ai_base.trainable = True
lr = keras.optimizers.schedules.CosineDecay(1e-4, decay_steps=len(train2) * EPOCHS_R)
ai_resnet.compile(optimizer=keras.optimizers.AdamW(lr, weight_decay=1e-4),
                  loss="binary_crossentropy", metrics=["accuracy"])

cb_r2 = [keras.callbacks.ModelCheckpoint(f"{SAVE_DIR}/resnet50_cifake_best.keras",
                                         monitor="val_accuracy", save_best_only=True),
         keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3,
                                       restore_best_weights=True)]
a2 = ai_resnet.fit(train2, validation_data=val2, epochs=EPOCHS_R, callbacks=cb_r2, verbose=2)

print("ResNet50 CIFAKE test acc: {:.2f}%".format(ai_resnet.evaluate(test2, verbose=0)[1] * 100))
evaluate_binary(ai_resnet, test2)
ai_resnet.save(f"{SAVE_DIR}/resnet50_ai_image_detector.keras")

In [ ]:
def make_grad_model_resnet(m, base_name="resnet50"):
    return keras.Model(m.inputs, [m.get_layer(base_name).output, m.output])

MODELS = {
    "Deepfake face detector": (face_resnet, make_grad_model_resnet(face_resnet), (128, 128)),
    "AI-generated image detector": (ai_resnet, make_grad_model_resnet(ai_resnet), (32, 32)),
}

In [ ]:
def jpeg_report(m, name):
    for q in [None, 95, 75, 50]:
        ds = test_mix if q is None else jpeg_ds(test_mix, q)
        y = np.concatenate([b[1].numpy().ravel() for b in ds]).astype(int)
        p = (m.predict(ds, verbose=0).ravel() > 0.5).astype(int)
        print(f"{name:22s} JPEG {str(q or 'none'):>4}: fake caught {(p[y==0]==0).mean()*100:5.1f}% | "
              f"real kept {(p[y==1]==1).mean()*100:5.1f}%")

scratch_plain = keras.models.load_model(f"{SAVE_DIR}/deepfake_detector.keras", compile=False)
jpeg_report(scratch_plain, "scratch")
jpeg_report(model, "scratch + JPEG aug")      # `model` is jpeg_robust from cell 22
jpeg_report(face_resnet, "ResNet50")

In [ ]:
face_base.trainable = True
EPOCHS_RJ = 4
lr = keras.optimizers.schedules.CosineDecay(5e-5, decay_steps=len(train_ds) * EPOCHS_RJ)
face_resnet.compile(optimizer=keras.optimizers.AdamW(lr, weight_decay=1e-4),
                    loss="binary_crossentropy", metrics=["accuracy"])

cb_rj = [keras.callbacks.ModelCheckpoint(f"{SAVE_DIR}/resnet50_jpeg_robust.keras",
                                         monitor="val_accuracy", save_best_only=True),
         keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2,
                                       restore_best_weights=True)]
face_resnet.fit(train_aug, validation_data=val_aug, epochs=EPOCHS_RJ,
                callbacks=cb_rj, verbose=2)

jpeg_report(face_resnet, "ResNet50 + JPEG aug")

In [ ]:
//Hallucination//

!pip install -q transformers datasets

import os, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             roc_auc_score, confusion_matrix, classification_report,
                             precision_recall_curve)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, DataCollatorWithPadding)

torch.manual_seed(42); np.random.seed(42)
print("GPU:", torch.cuda.is_available())   # must be True

In [ ]:
from google.colab import files
if not os.path.exists("train.csv"):
    files.upload()                      # choose train.csv
df = pd.read_csv("train.csv")
print(df.shape, list(df.columns))

col = {c.lower(): c for c in df.columns}
P, A, T = col["prompt"], col["answer"], col["target"]
df = df[[P, A, T]].rename(columns={P: "prompt", A: "answer", T: "label"}).dropna()
df["label"] = df["label"].astype(int)

counts = df["label"].value_counts().sort_index()
print(counts, f"\nhallucinated = {counts[1]/len(df)*100:.1f}%")   # about 5.4%
counts.plot.bar(title="Class balance (0 = fine, 1 = hallucinated)"); plt.show()

In [ ]:
train_df, val_df = train_test_split(df, test_size=0.15, stratify=df["label"], random_state=42)
print(len(train_df), len(val_df), "val hallucinated:", int(val_df["label"].sum()))

In [ ]:
MODEL = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)

class PairDS(torch.utils.data.Dataset):
    def __init__(self, d):
        self.enc = tok(list(d["prompt"]), list(d["answer"]), truncation=True, max_length=256)
        self.y = d["label"].tolist()
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.y[i]
        return item

train_ds, val_ds = PairDS(train_df), PairDS(val_df)
print(tok.decode(train_ds[0]["input_ids"][:60]))

In [ ]:
y = train_df["label"].values
n = len(y)
w = np.array([n / (2 * (y == 0).sum()), n / (2 * (y == 1).sum())])
print("class weights:", w.round(2))
class_w = torch.tensor(w, dtype=torch.float)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels,
                                                 weight=class_w.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def metrics(p):
    logits, labels = p
    prob = torch.softmax(torch.tensor(logits), -1)[:, 1].numpy()
    pred = logits.argmax(-1)
    pr, rc, f1, _ = precision_recall_fscore_support(labels, pred, average="binary", zero_division=0)
    return {"accuracy": accuracy_score(labels, pred), "precision": pr, "recall": rc,
            "f1": f1, "auc": roc_auc_score(labels, prob)}

args = TrainingArguments(
    output_dir="halluc_out", num_train_epochs=3, per_device_train_batch_size=16,
    per_device_eval_batch_size=64, learning_rate=2e-5, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1",
    fp16=torch.cuda.is_available(), report_to="none", seed=42)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

In [ ]:
log = pd.DataFrame(trainer.state.log_history)
ev = log[log["eval_accuracy"].notna()]
plt.plot(ev["epoch"], ev["eval_accuracy"] * 100, marker="o")
plt.xlabel("Epoch"); plt.ylabel("Validation accuracy (%)"); plt.title("Accuracy per epoch"); plt.show()
ev[["epoch", "eval_accuracy", "eval_precision", "eval_recall", "eval_f1", "eval_auc"]]

In [ ]:
out = trainer.predict(val_ds)
logits, y_true = out.predictions, out.label_ids
y_prob = torch.softmax(torch.tensor(logits), -1)[:, 1].numpy()
y_pred = logits.argmax(-1)

print(classification_report(y_true, y_pred, target_names=["not halluc.", "halluc."], digits=3))
print("ROC-AUC:", round(roc_auc_score(y_true, y_prob), 3))

cm = confusion_matrix(y_true, y_pred)
print(cm)
print(f"caught {cm[1,1]} of {cm[1].sum()}, missed {cm[1,0]}, false alarms {cm[0,1]}")

baseline = (y_true == 0).mean() * 100
print(f"Always say 'not hallucinated': {baseline:.1f}% accuracy, 0 caught")
print(f"Our model: {accuracy_score(y_true, y_pred)*100:.1f}% accuracy")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax[0].text(j, i, cm[i, j], ha="center", va="center",
                   color="white" if cm[i, j] > cm.max()/2 else "black")
ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(["not halluc.", "halluc."])
ax[0].set_yticks([0, 1]); ax[0].set_yticklabels(["not halluc.", "halluc."])
ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("Actual"); ax[0].set_title("Confusion matrix")
ax[1].bar(["Always say no", "DistilBERT"], [baseline, accuracy_score(y_true, y_pred) * 100])
ax[1].set_ylim(0, 100); ax[1].set_title("Accuracy alone looks fine for both")
plt.show()

In [ ]:
pr, rc, th = precision_recall_curve(y_true, y_prob)
f1s = 2 * pr * rc / (pr + rc + 1e-9)
b = int(np.argmax(f1s[:-1]))
print(f"best threshold {th[b]:.2f}: precision {pr[b]:.2f}, recall {rc[b]:.2f}, F1 {f1s[b]:.2f}")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
trainer.save_model("/content/drive/MyDrive/deepfake/halluc_model")
tok.save_pretrained("/content/drive/MyDrive/deepfake/halluc_model")

In [ ]:
!pip install -q gradio transformers
import os, numpy as np, tensorflow as tf, torch, matplotlib.pyplot as plt
for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)     # leave GPU memory for PyTorch
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image
import gradio as gr
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSequenceClassification

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/deepfake"
print(os.listdir(SAVE_DIR))

In [ ]:
face_model = keras.models.load_model(f"{SAVE_DIR}/jpeg_robust.keras", compile=False)
ai_model   = keras.models.load_model(f"{SAVE_DIR}/ai_image_detector.keras", compile=False)

dev = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(f"{SAVE_DIR}/halluc_model")
bert = AutoModelForSequenceClassification.from_pretrained(f"{SAVE_DIR}/halluc_model").to(dev).eval()

In [ ]:
def make_grad_model(m):
    last_relu = [l for l in m.layers if isinstance(l, layers.ReLU)][-1]
    return keras.Model(m.inputs, [last_relu.output, m.output])

MODELS = {
    "Deepfake face detector": (make_grad_model(face_model), (128, 128)),
    "AI-generated image detector": (make_grad_model(ai_model), (32, 32)),
}

def gradcam_for(gm, arr, size):
    x = tf.expand_dims(tf.cast(arr, tf.float32), 0)
    with tf.GradientTape() as tape:
        conv, pred = gm(x, training=False)
        p = pred[:, 0]
        score = p if p[0] > 0.5 else 1.0 - p
    grads = tf.cast(tape.gradient(score, conv), tf.float32)
    conv = tf.cast(conv, tf.float32)
    w = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(w * conv, axis=-1))[0]
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return tf.image.resize(cam[..., None], size).numpy()[..., 0], float(p[0])

In [ ]:
def detect_image(image, which):
    if image is None:
        return None, None
    gm, size = MODELS[which]
    pil = Image.fromarray(image).convert("RGB")
    arr = np.array(pil.resize(size, Image.BILINEAR)).astype("float32")
    cam, p_real = gradcam_for(gm, arr, size)
    base = np.array(pil.resize((384, 384))) / 255.0
    cam_big = np.array(Image.fromarray((cam * 255).astype("uint8")).resize((384, 384))) / 255.0
    heat = plt.cm.jet(cam_big)[..., :3]
    overlay = (0.55 * base + 0.45 * heat).clip(0, 1)
    verdict = {"FAKE (AI-generated)": 1 - p_real, "REAL": p_real}
    return verdict, (overlay * 255).astype("uint8")

def detect_text(prompt, answer):
    if not (prompt or "").strip() or not (answer or "").strip():
        return None
    enc = tok(prompt, answer, truncation=True, max_length=256, return_tensors="pt").to(dev)
    with torch.no_grad():
        p = torch.softmax(bert(**enc).logits, -1)[0].cpu().numpy()
    return {"Likely HALLUCINATED": float(p[1]), "Looks OK": float(p[0])}

In [ ]:
with gr.Blocks(title="Untrustworthy AI Content Detector") as demo:
    gr.Markdown("# Untrustworthy AI Content Detector\nDeepfake images and LLM hallucinations in one demo.")
    with gr.Tab("Image: deepfake / AI-generated"):
        with gr.Row():
            with gr.Column():
                img = gr.Image(type="numpy", label="Upload an image")
                which = gr.Dropdown(list(MODELS), value="Deepfake face detector", label="Model")
                b1 = gr.Button("Check image")
            with gr.Column():
                out_lbl = gr.Label(label="Verdict")
                out_img = gr.Image(label="Where the model looked (Grad-CAM)")
        b1.click(detect_image, [img, which], [out_lbl, out_img])
    with gr.Tab("Text: LLM hallucination"):
        prompt = gr.Textbox(label="Prompt given to the LLM", lines=2)
        answer = gr.Textbox(label="LLM's answer", lines=4)
        b2 = gr.Button("Check answer")
        out_txt = gr.Label(label="Verdict")
        b2.click(detect_text, [prompt, answer], out_txt)
        gr.Examples([["Who wrote Hamlet?", "Hamlet was written by William Shakespeare around 1600."],
                     ["Who was the first person on the Moon?", "Yuri Gagarin walked on the Moon in 1969."]],
                    [prompt, answer])
    gr.Markdown("Text model: class-weighted DistilBERT, F1 0.34 on the hallucinated class. "
                "It flags answers to review, it does not prove they are wrong.")
demo.launch(share=True)